In [0]:
%run ./00_config


In [0]:
import io, os, zipfile, urllib.request
target_dir = f"{LANDING}/{run_date}"
url = f"https://www150.statcan.gc.ca/n1/tbl/csv/{table_id}-eng.zip"

try:
    with urllib.request.urlopen(url, timeout=120) as response:
        data = response.read()
    os.makedirs(target_dir, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(data)) as z:
        name = f"{table_id}.csv"          # keep only the data file, not the metadata file
        with z.open(name) as src, open(f"{target_dir}/{name}", "wb") as dst:
            dst.write(src.read())
    print(f"Downloaded {len(data) / 1e6:.1f} MB into {target_dir}")

except Exception as e:
    print("Download did not work:", repr(e)[:200])
    try:
        folders = sorted(f.name for f in dbutils.fs.ls(LANDING))
    except Exception:
        folders = []
    if not folders:
        raise Exception("No download and the landing folder is empty. "
                        "Upload the CSV by hand (step 2.2), then run again.")
    print("Continuing with the files already in landing. Latest folder:", folders[-1])